
# Transformers (BERT, GPT) — Notebook de estudo

Este notebook foi montado para te dar uma visão **teórica + prática leve** sobre:

- o que são Transformers
- como funciona **self-attention**
- diferença entre **BERT** e **GPT**
- uso prático com `transformers`
- embeddings com BERT para comparar frases

> Ideia: entender bem o conceito agora e depois plugar nos seus próprios dados.



## 1) Setup

Se estiver rodando localmente e faltar alguma biblioteca, descomente a célula abaixo.


In [1]:

# Descomente se precisar instalar
!pip install transformers torch sentence-transformers scikit-learn pandas numpy matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 14.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [sentence-transformers]ence-transformers]

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:

import math
import numpy as np
import pandas as pd
from pprint import pprint



## 2) Visão geral

Antes dos Transformers, modelos como **RNN** e **LSTM** processavam texto de forma sequencial.

Exemplo da frase:

> "O banco aprovou o crédito"

A palavra **banco** pode significar:
- instituição financeira
- banco de sentar

O Transformer melhora isso porque olha o contexto inteiro ao mesmo tempo.

### Ideia central
O mecanismo principal é o **self-attention**:

Cada token pergunta algo como:

> "Quais outros tokens desta frase são importantes para eu me representar melhor?"



## 3) Self-Attention na prática (versão mini)

A fórmula clássica é:

\[
Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

Onde:
- **Q (Query)**: o que o token está procurando
- **K (Key)**: o que cada token oferece
- **V (Value)**: o conteúdo de cada token

Abaixo, vamos montar um exemplo pequeno em NumPy.


In [3]:

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

# Exemplo simples: 3 tokens, dimensão 4
Q = np.array([
    [1.0, 0.0, 1.0, 0.0],   # token 1
    [0.0, 1.0, 0.0, 1.0],   # token 2
    [1.0, 1.0, 0.0, 0.0],   # token 3
])

K = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 0.0],
])

V = np.array([
    [10.0, 0.0],   # valor token 1
    [0.0, 10.0],   # valor token 2
    [5.0, 5.0],    # valor token 3
])

dk = K.shape[1]
scores = (Q @ K.T) / math.sqrt(dk)
weights = softmax(scores, axis=-1)
attention_output = weights @ V

print("Scores:")
print(scores)
print("\nPesos de atenção:")
print(weights)
print("\nSaída final:")
print(attention_output)


Scores:
[[1.  0.  0.5]
 [0.  1.  0.5]
 [0.5 0.5 1. ]]

Pesos de atenção:
[[0.50648039 0.18632372 0.30719589]
 [0.18632372 0.50648039 0.30719589]
 [0.27406862 0.27406862 0.45186276]]

Saída final:
[[6.60078334 3.39921666]
 [3.39921666 6.60078334]
 [5.         5.        ]]



### Leitura do resultado

- Cada linha em **pesos de atenção** mostra quanto um token presta atenção nos outros.
- A **saída final** é uma combinação ponderada dos vetores `V`.

Isso é o coração do Transformer.



## 4) BERT vs GPT

### BERT
- usa arquitetura baseada em **Encoder**
- lê o contexto dos dois lados
- ótimo para **entendimento**
- tarefas comuns:
  - classificação
  - análise de sentimento
  - NER
  - embeddings semânticos

### GPT
- usa arquitetura baseada em **Decoder**
- lê da esquerda para a direita
- ótimo para **geração**
- tarefas comuns:
  - chat
  - geração de texto
  - resumo
  - código

### Regra de bolso
- **BERT**: "entender"
- **GPT**: "continuar / gerar"



## 5) Exemplo com BERT para classificação

Vamos usar um pipeline pronto do Hugging Face.


In [4]:

from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

texts = [
    "Esse notebook ficou muito bom.",
    "Achei o resultado confuso.",
    "O modelo parece promissor para produção."
]

results = classifier(texts)
for text, result in zip(texts, results):
    print(f"Texto: {text}")
    print(f"Resultado: {result}")
    print("-" * 60)


/Users/yandrade/PycharmProjects/ia_study/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19858.08it/s]


Texto: Esse notebook ficou muito bom.
Resultado: {'label': '5 stars', 'score': 0.5938502550125122}
------------------------------------------------------------
Texto: Achei o resultado confuso.
Resultado: {'label': '2 stars', 'score': 0.37266919016838074}
------------------------------------------------------------
Texto: O modelo parece promissor para produção.
Resultado: {'label': '3 stars', 'score': 0.4720689058303833}
------------------------------------------------------------



## 6) Exemplo com GPT para geração de texto

Aqui a ideia é simples: dar um começo e deixar o modelo continuar.


In [5]:

generator = pipeline("text-generation", model="gpt2")

prompt = "Transformers revolucionaram o NLP porque"
generated = generator(
    prompt,
    max_length=60,
    num_return_sequences=1,
    truncation=True
)

pprint(generated)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3627.53it/s]
Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Transformers revolucionaram o NLP porque estas. In nomee '
                    'quando de la nomee estas, de serra una nuestro el donde '
                    'conseguentes.\n'
                    '\n'
                    'JANUARY 13, 2013\n'
                    '\n'
                    'BARTON, MA. (AP) — A Massachusetts court ruled that a '
                    'woman who was accused of raping a 12-year-old girl could '
                    "not prove she was a victim because she wasn't a victim, "
                    "after the judge ruled that the state couldn't prove that "
                    "because her DNA matched the child's DNA to an accused "
                    'rapist.\n'
                    '\n'
                    'The case was raised by the prosecution and defense team, '
                    'who said the case does not look like a rape and that the '
                    "girl's DNA matched the girl's DNA to the rapist's.\n"
                    '\n'
  


## 7) Embeddings com BERT/Sentence Transformers

Agora uma parte bem útil no mundo real:
transformar frases em vetores e medir similaridade semântica.

Isso serve muito para:
- busca semântica
- deduplicação
- matching de texto
- FAQ inteligente
- agrupamento de descrições parecidas


In [6]:

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sentences = [
    "O usuário solicitou um adiantamento salarial.",
    "O cliente pediu antecipação do salário.",
    "Hoje está chovendo bastante em São Paulo.",
    "A conta foi aprovada com sucesso."
]

embeddings = model.encode(sentences)

sim_matrix = cosine_similarity(embeddings)
sim_df = pd.DataFrame(sim_matrix, index=sentences, columns=sentences)
sim_df


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13996.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,O usuário solicitou um adiantamento salarial.,O cliente pediu antecipação do salário.,Hoje está chovendo bastante em São Paulo.,A conta foi aprovada com sucesso.
O usuário solicitou um adiantamento salarial.,1.000000,0.601700,0.409977,0.414445
O cliente pediu antecipação do salário.,0.601700,1.000000,0.542422,0.548191
Hoje está chovendo bastante em São Paulo.,0.409977,0.542422,1.000000,0.381399
A conta foi aprovada com sucesso.,0.414445,0.548191,0.381399,1.000000



### O que observar
As frases com significado parecido tendem a ter similaridade maior.

Exemplo esperado:
- "adiantamento salarial"
- "antecipação do salário"

Essas duas devem ficar bem mais próximas entre si do que da frase sobre chuva.



## 8) Teste com frases próprias

Troque as frases abaixo por exemplos do seu contexto.


In [7]:

custom_sentences = [
    "Pagamento aprovado.",
    "Transação concluída com sucesso.",
    "Erro ao processar o cartão.",
    "Falha no pagamento do usuário."
]

custom_embeddings = model.encode(custom_sentences)
custom_sim = cosine_similarity(custom_embeddings)

pd.DataFrame(custom_sim, index=custom_sentences, columns=custom_sentences)


,Pagamento aprovado.,Transação concluída com sucesso.,Erro ao processar o cartão.,Falha no pagamento do usuário.
Pagamento aprovado.,1.000000,0.442158,0.482360,0.746807
Transação concluída com sucesso.,0.442158,1.000000,0.440664,0.402492
Erro ao processar o cartão.,0.482360,0.440664,1.000000,0.451059
Falha no pagamento do usuário.,0.746807,0.402492,0.451059,1.000000



## 9) Mini projeto aplicado no seu cenário

Se você quiser conectar com dados reais anonimizados, aqui vão algumas ideias muito boas:

### A) Similaridade de descrições
Exemplo:
- descrições de erro
- mensagens de logs
- motivos de recusa
- observações operacionais

Objetivo:
- encontrar mensagens parecidas
- agrupar padrões
- detectar duplicidades

### B) Classificação de texto
Exemplo:
- categorizar tickets
- classificar tipo de erro
- identificar prioridade
- rotular motivo de fraude, NSF, salary issue etc.

### C) Busca semântica
Exemplo:
- buscar casos parecidos
- procurar FAQs próximas
- encontrar registros relevantes sem depender só de palavra exata



## 10) Exemplo de estrutura para plugar CSV

Abaixo um esqueleto para você adaptar depois.


In [ ]:

# Exemplo de estrutura
# df = pd.read_csv("seu_arquivo.csv")
# df.head()

# Suponha que exista uma coluna de texto
# text_col = "description"

# embeddings = model.encode(df[text_col].fillna("").tolist(), show_progress_bar=True)
# sim_matrix = cosine_similarity(embeddings)

# Para encontrar textos mais parecidos com o primeiro registro:
# idx = 0
# similarities = sim_matrix[idx]
# nearest = np.argsort(similarities)[::-1][1:6]
# df.iloc[nearest]



## 11) Resumo final

### Transformer
- usa atenção para relacionar tokens da sequência inteira

### BERT
- melhor para **entendimento**
- embeddings e classificação

### GPT
- melhor para **geração**
- continua texto token a token

### Aplicações práticas
- classificação
- busca semântica
- comparação entre frases
- assistentes
- sumarização
- geração de conteúdo

---

## 12) Próximo nível de estudo

Quando quiser continuar, o passo natural é:

1. **Fine-tuning de BERT**
2. **Embeddings em base real**
3. **Attention mask / positional encoding**
4. **LLMs, prompting e RAG**

Aí o bicho fica elegante de verdade.
